In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

In [31]:
# Check missing values, duplicates, and outliers
df_clean = pd.read_csv(r'../data/sample_cleaned.csv')

# Code to output only columns with missing values
    # missing value: no value at position where data should exist
missing_cols = df_clean.isnull().sum()
missing_cols = missing_cols[missing_cols > 0]  # Keep only columns with at least 1 missing value

print("=== Columns with Missing Values ===")
if missing_cols.empty:
    print("No missing values in the dataset!")
else:
    print(missing_cols)

# 2. Classify columns by data type
print("\n=== Columns by Data Type (Auto-detected) ===")

# Get list of all dtypes present in dataset
unique_dtypes = df_clean.dtypes.unique()

# Classify and output columns by each dtype
for dtype in unique_dtypes:
    cols = df_clean.columns[df_clean.dtypes == dtype].tolist()
    print(f"\n{dtype} columns ({len(cols)}):")
    print(cols)

# 3. Check non-numeric columns in detail
print("\n=== Non-numeric Columns Detail ===")
non_numeric_cols = df_clean.select_dtypes(exclude=['int64', 'float64']).columns

for col in non_numeric_cols:
    print(f"\n[{col}]")
    print(f"  Data type: {df_clean[col].dtype}")
    print(f"  Unique values: {df_clean[col].nunique()}")
    print(f"  Sample values: {df_clean[col].unique()[:5]}")  # First 5 only

# 4. Find object type columns that are actually numeric
print("\n=== Object columns that might be numeric ===")
for col in non_numeric_cols:
    try:
        # Attempt to convert to numeric
        pd.to_numeric(df_clean[col], errors='raise')
        print(f"{col} → Can be converted to numeric!")
    except:
        print(f"{col} → Contains non-numeric values")

# 5. Data type summary table (more readable)
print("\n=== Data Type Summary Table ===")
dtype_summary = pd.DataFrame({
    'Column': df_clean.columns,
    'Data Type': df_clean.dtypes.values,
    'Non-Null Count': df_clean.count().values,
    'Null Count': df_clean.isnull().sum().values,
    'Unique Values': [df_clean[col].nunique() for col in df_clean.columns]
})
print(dtype_summary.to_string(index=False))

# Reset settings to default
pd.reset_option('display.max_rows')

=== Columns with Missing Values ===
Flow Byts/s    112
dtype: int64

=== Columns by Data Type (Auto-detected) ===

int64 columns (42):
['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Pkt Len Min', 'Pkt Len Max', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Fwd Seg Size Min']

object columns (2):
['Timestamp', 'Label']

float64 columns (36):
['Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len M

Existing data types: int, float, object

Flow Byts/s has missing values
=> Number of missing values: 112, dtype: int64

dtype: object -> ['Timestamp', 'Label']

In [ ]:
# Check infinite/NaN values
print("\n=== Inf values ===")
print("Inf values:", df_clean.isin([np.inf, -np.inf]).sum().sum())

# Recheck sample count by attack type
print("\n=== Attack Type Distribution ===")
print(df_clean['Label'].value_counts())

# Check duplicate rows
print("\n=== Duplicate Rows ===")
print(f"Number of duplicates: {df_clean.duplicated().sum()}") 
print()

# duplicated=True → returns all duplicate rows
duplicate_rows = df_clean[df_clean.duplicated(keep=False)]

# Check label distribution of duplicate rows
print("=== Labels among duplicate rows ===")
print(duplicate_rows['Label'].value_counts())


=== Inf values ===
Inf values: 246

=== Attack Type Distribution ===
Label
Benign                      14325
DoS attacks-GoldenEye        2954
DDoS attacks-LOIC-HTTP       2936
FTP-BruteForce               2905
DoS attacks-SlowHTTPTest     2902
Bot                          1659
DDOS attack-LOIC-UDP         1390
Brute Force -Web              611
Brute Force -XSS              230
SQL Injection                  87
Name: count, dtype: int64

=== Duplicate Rows ===
Number of duplicates: 4606

=== Labels among duplicate rows ===
Label
FTP-BruteForce              2607
DoS attacks-SlowHTTPTest    2582
Benign                        24
Name: count, dtype: int64


24 Benign duplicate entries exist -> delete

In [32]:
df_clean = pd.read_csv(r'../data/sample_cleaned.csv')

print("=== Before removing Benign duplicates ===")
print(f"Total rows: {len(df_clean)}")
print(f"Benign rows: {(df_clean['Label'] == 'Benign').sum()}")

# Remove only Benign duplicates (keep first occurrence)
benign_mask = df_clean['Label'] == 'Benign'
benign_duplicated = df_clean[benign_mask].duplicated(keep='first')

# Remove only rows that are Benign and duplicated
df_final = df_clean[~(benign_mask & benign_duplicated)].copy()

print("\n=== After removing Benign duplicates ===")
print(f"Total rows: {len(df_final)}")
print(f"Benign rows: {(df_final['Label'] == 'Benign').sum()}")
print(f"Removed rows: {len(df_clean) - len(df_final)}")

# Save
df_final.to_csv(output_path, index=False)

=== Before removing Benign duplicates ===
Total rows: 29999
Benign rows: 14325

=== After removing Benign duplicates ===
Total rows: 29987
Benign rows: 14313
Removed rows: 12


## Inf Processing


In [34]:
# Load data
df = pd.read_csv(r'../data/02_EDA_delete_Benign.csv')

# Find columns with Inf values

# Count Inf values per column
inf_counts = df.isin([np.inf, -np.inf]).sum()
inf_columns = inf_counts[inf_counts > 0].sort_values(ascending=False)

print("\n=== Columns with Inf values ===")
for col, count in inf_columns.items():
    percentage = (count / len(df)) * 100
    print(f"{col:30s}: {count:4d} ({percentage:.2f}%)")

print(f"\nTotal Inf values: {inf_counts.sum()}")
print(f"Number of affected columns: {len(inf_columns)}")


=== Columns with Inf values ===
Flow Pkts/s                   :  179 (0.60%)
Flow Byts/s                   :   67 (0.22%)

Total Inf values: 246
Number of affected columns: 2


In [35]:
# Analyze Inf causes

# Extract rows with Inf
inf_rows = df[df.isin([np.inf, -np.inf]).any(axis=1)].copy()
print(f"\nNumber of rows containing Inf: {len(inf_rows)}")

# Analyze cause for each Inf column
for col in inf_columns.index:
    print(f"\n--- [{col}] Analysis ---")
    
    # Rows with Inf in this column
    col_inf_rows = df[df[col].isin([np.inf, -np.inf])]
    
    # 1. Check Flow Duration (most common cause)
    if 'Flow Duration' in df.columns:
        duration_zero = (col_inf_rows['Flow Duration'] == 0).sum()
        duration_very_small = (col_inf_rows['Flow Duration'] < 100).sum()
        print(f"  Flow Duration = 0: {duration_zero}")
        print(f"  Flow Duration < 100: {duration_very_small}")
        
        if duration_zero > 0:
            print(f"  → Inf in {col} is mainly due to Division by Zero!")
    
    # 2. Check other features related to this column
    # Example: Flow Byts/s is related to Total Bytes and Flow Duration
    if 'Byts/s' in col or 'Pkts/s' in col:
        print(f"  Cause: Denominator is 0 or very small in ratio calculation (numerator/denominator)")
    
    # 3. Check Label distribution (which attacks have more occurrences?)
    print(f"\n  Label distribution:")
    label_dist = col_inf_rows['Label'].value_counts()
    for label, count in label_dist.items():
        print(f"    {label}: {count}")


Number of rows containing Inf: 179

--- [Flow Pkts/s] Analysis ---
  Flow Duration = 0: 179
  Flow Duration < 100: 179
  → Inf in Flow Pkts/s is mainly due to Division by Zero!
  Cause: Denominator is 0 or very small in ratio calculation (numerator/denominator)

  Label distribution:
    Benign: 179

--- [Flow Byts/s] Analysis ---
  Flow Duration = 0: 67
  Flow Duration < 100: 67
  → Inf in Flow Byts/s is mainly due to Division by Zero!
  Cause: Denominator is 0 or very small in ratio calculation (numerator/denominator)

  Label distribution:
    Benign: 67


According to code results, labels with Inf values are only benign --> deletion can be considered

In [36]:
# Analyze Inf patterns

# Check main features of rows with Inf
print("\n=== Main characteristics of Inf rows ===")
key_features = ['Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 
                'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Label']

# Select only existing columns
existing_features = [f for f in key_features if f in inf_rows.columns]
print(inf_rows[existing_features].describe())

# Compare with normal rows without Inf
normal_rows = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
print("\n=== Main characteristics of normal rows (for comparison) ===")
print(normal_rows[existing_features].describe())


=== Main characteristics of Inf rows ===
       Flow Duration  Tot Fwd Pkts  Tot Bwd Pkts  TotLen Fwd Pkts  \
count          179.0         179.0         179.0       179.000000   
mean             0.0           2.0           0.0        12.491620   
std              0.0           0.0           0.0        17.675403   
min              0.0           2.0           0.0         0.000000   
25%              0.0           2.0           0.0         0.000000   
50%              0.0           2.0           0.0         0.000000   
75%              0.0           2.0           0.0        31.000000   
max              0.0           2.0           0.0       107.000000   

       TotLen Bwd Pkts  
count            179.0  
mean               0.0  
std                0.0  
min                0.0  
25%                0.0  
50%                0.0  
75%                0.0  
max                0.0  

=== Main characteristics of normal rows (for comparison) ===
       Flow Duration   Tot Fwd Pkts  Tot Bwd Pkts

Comparison of values between inf rows and normal rows

inf: Main characteristics are close to 0

normal: Main characteristic values are large


=> Benign with inf is extremely short and small traffic

In [37]:
# Evaluate processing methods

# Method 1: Median replacement
print("\n[Method 1] Inf → NaN → Median replacement")
df_method1 = df.copy()
df_method1.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check median value of each column
print("\nValues to be replaced (median):")
for col in inf_columns.index:
    median_val = df_method1[col].median()
    print(f"  {col:30s}: {median_val:.2f}")

# Method 2: Row deletion
print(f"\n[Method 2] Remove rows containing Inf")
print(f"  Number of rows to be removed: {len(inf_rows)} / {len(df)} ({len(inf_rows)/len(df)*100:.2f}%)")
print(f"  Number of remaining rows: {len(normal_rows)}")

# Label distribution change when removed
print("\n  Label distribution after removal:")
print(normal_rows['Label'].value_counts())

# Method 3: Customized processing by feature
print("\n[Method 3] Customized processing by feature")
print("  - Flow Duration = 0 → Change to 1 (very short flow)")
print("  - Cap ratio features with maximum value")


[Method 1] Inf → NaN → Median replacement

Values to be replaced (median):
  Flow Pkts/s                   : 17.53
  Flow Byts/s                   : 33.53

[Method 2] Remove rows containing Inf
  Number of rows to be removed: 179 / 29987 (0.60%)
  Number of remaining rows: 29808

  Label distribution after removal:
Label
Benign                      14134
DoS attacks-GoldenEye        2954
DDoS attacks-LOIC-HTTP       2936
FTP-BruteForce               2905
DoS attacks-SlowHTTPTest     2902
Bot                          1659
DDOS attack-LOIC-UDP         1390
Brute Force -Web              611
Brute Force -XSS              230
SQL Injection                  87
Name: count, dtype: int64

[Method 3] Customized processing by feature
  - Flow Duration = 0 → Change to 1 (very short flow)
  - Cap ratio features with maximum value


Median replacement -> Originally infinite values per second, but 17 and 33 are too low to replace

Even after row deletion, sufficient Benign values exist ==> Delete

In [39]:
# Load data
input_path = r'../data/02_EDA_delete_Benign.csv'
df = pd.read_csv(input_path)

print("Delete Inf rows and save")

print("\n=== Before processing ===")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Check Inf values
inf_mask = df.isin([np.inf, -np.inf]).any(axis=1)
inf_count = inf_mask.sum()
print(f"Rows containing Inf: {inf_count}")

# Check Label of rows containing Inf
if inf_count > 0:
    inf_rows = df[inf_mask]
    print(f"\nLabel distribution of Inf rows:")
    print(inf_rows['Label'].value_counts())

# Delete Inf rows
df_cleaned = df[~inf_mask].copy()

print("\n=== After processing ===")
print(f"Total rows: {len(df_cleaned)}")
print(f"Removed rows: {inf_count} ({inf_count/len(df)*100:.2f}%)")

# Check Label distribution
print(f"\nLabel distribution:")
print(df_cleaned['Label'].value_counts())

# Final validation
print("\n=== Final validation ===")
print(f"Remaining Inf values: {df_cleaned.isin([np.inf, -np.inf]).sum().sum()}")
print(f"Remaining NaN values: {df_cleaned.isnull().sum().sum()}")

# Save
output_path = r'../data/02_EDA_inf.csv'
df_cleaned.to_csv(output_path, index=False)

print(f"Processing complete!")
print(f"Final Shape: {df_cleaned.shape}")

Delete Inf rows and save

=== Before processing ===
Total rows: 29987
Total columns: 80
Rows containing Inf: 179

Label distribution of Inf rows:
Label
Benign    179
Name: count, dtype: int64

=== After processing ===
Total rows: 29808
Removed rows: 179 (0.60%)

Label distribution:
Label
Benign                      14134
DoS attacks-GoldenEye        2954
DDoS attacks-LOIC-HTTP       2936
FTP-BruteForce               2905
DoS attacks-SlowHTTPTest     2902
Bot                          1659
DDOS attack-LOIC-UDP         1390
Brute Force -Web              611
Brute Force -XSS              230
SQL Injection                  87
Name: count, dtype: int64

=== Final validation ===
Remaining Inf values: 0
Remaining NaN values: 0
Processing complete!
Final Shape: (29808, 80)


## Timestamp Processing


For Timestamp, will leave it for now<br>
Not used in rule design -> Automatically excluded during numerical analysis<br>
Can be useful later for visualization

=> Conclusion: Preserve data!